# Obesity Classification — Complete Scientific Analysis
## Raw Survey Data → Honest Predictive Model (No Data Leakage)

**What this notebook covers:**
1. Why predicting obesity from lifestyle is scientifically interesting (and hard)
2. The data leakage trap — and how to avoid it
3. Full EDA of every feature: what it measures, example, chart, how to read the chart
4. Data cleaning with step-by-step reasoning
5. Feature engineering with formulas and rationale
6. Model training, cross-validation, hyperparameter tuning
7. Complete evaluation: every metric explained

**The research question:**
Can we predict a person's obesity level from *lifestyle and behavioral data alone* —
without knowing their weight or height — and if so, *which behaviors matter most?*

---

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, f1_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
SEED = 42; np.random.seed(SEED)

DATA_PATH    = "../models/ObesityDataSet_raw_and_data_sinthetic.csv"
TARGET       = "NObeyesdad"
TARGET_ORDER = {"Insufficient_Weight":0,"Normal_Weight":1,"Overweight_Level_I":2,
                "Overweight_Level_II":3,"Obesity_Type_I":4,"Obesity_Type_II":5,"Obesity_Type_III":6}
TARGET_LABELS = list(TARGET_ORDER.keys())
PALETTE       = sns.color_palette("viridis", n_colors=7)
FREQ_MAP      = {"no":0,"Sometimes":1,"Frequently":2,"Always":3}
SMOTE_CLIPS   = {"FCVC":(1,3),"NCP":(1,4),"CH2O":(1,3),"FAF":(0,3),"TUE":(0,2)}
print("Libraries loaded successfully.")

---
## Chapter 1 — Why We Drop Weight and Height First

### The Data Leakage Problem

**What is data leakage?**
Data leakage occurs when information that *defines* the target variable is also used as a
training feature. The model learns to reverse-engineer the label rather than find genuine patterns.

**How does it apply here?**

The target variable `NObeyesdad` has 7 classes defined by WHO BMI thresholds:

| Class | BMI threshold |
|-------|--------------|
| Insufficient_Weight | BMI < 18.5 |
| Normal_Weight | 18.5 ≤ BMI < 25 |
| Overweight_Level_I | 25 ≤ BMI < 27.5 |
| Overweight_Level_II | 27.5 ≤ BMI < 30 |
| Obesity_Type_I | 30 ≤ BMI < 35 |
| Obesity_Type_II | 35 ≤ BMI < 40 |
| Obesity_Type_III | BMI ≥ 40 |

**And BMI = Weight ÷ Height²**

So: `Weight + Height → BMI → NObeyesdad`. Including Weight and Height means the model
can trivially compute BMI and look up the class — achieving ~97% F1 with zero insight into
*why* people become obese.

**Example:**
> Person: Weight = 95 kg, Height = 1.70 m
> BMI = 95 ÷ (1.70²) = 95 ÷ 2.89 = **32.9** → Obesity_Type_I
> A model just needs to learn "if BMI > 30 → Obesity_Type_I" — trivial math, zero science.

**What we do instead:** Drop both columns on the very first line, before any chart, any
analysis, any cleaning. Then the model must learn from diet, exercise, habits, and genetics.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Original dataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

# ── Drop Weight and Height BEFORE anything else ─────────────────────────────
df = df_raw.drop(columns=["Weight","Height"])
assert "Weight" not in df.columns and "Height" not in df.columns
print(f"After dropping leakage columns: {df.shape[0]} rows × {df.shape[1]} columns")

print(f"\n{'Column':<42} {'Type':<12} {'Unique values / Range'}")
print("─"*75)
for col in df.columns:
    if col == TARGET: continue
    dtype = str(df[col].dtype)
    if dtype == "object":
        vals = ", ".join(df[col].unique()[:4])
        print(f"  {col:<40} {dtype:<12} {vals}")
    else:
        print(f"  {col:<40} {dtype:<12} [{df[col].min():.1f} – {df[col].max():.1f}]")

---
## Chapter 2 — Understanding the Target Variable

### What are the 7 obesity classes?

The dataset has 7 ordered classes (ordered = each class is "heavier" than the previous one):

| # | Class | Meaning |
|---|-------|---------|
| 0 | Insufficient_Weight | Underweight — BMI < 18.5 |
| 1 | Normal_Weight | Healthy weight |
| 2 | Overweight_Level_I | Mildly overweight |
| 3 | Overweight_Level_II | Moderately overweight |
| 4 | Obesity_Type_I | Obese class I |
| 5 | Obesity_Type_II | Obese class II |
| 6 | Obesity_Type_III | Severely obese |

**Why 7 classes instead of 2 (obese / not obese)?**
Finer granularity makes the problem harder but more useful clinically — distinguishing
Overweight_Level_I from Obesity_Type_II requires understanding different lifestyle patterns.

**What to look for in the chart:**
Count the height of each bar. Are they roughly equal? Unequal? Why?

In [ ]:
counts = df[TARGET].value_counts().reindex(TARGET_LABELS)
fig, ax = plt.subplots(figsize=(11,4))
bars = ax.bar(TARGET_LABELS, counts.values, color=PALETTE, edgecolor="white", linewidth=0.8)
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_ylabel("Number of rows"); ax.set_title("Target Variable Distribution  (n = 2,111)", fontsize=13, fontweight="bold")
ax.set_xticklabels(TARGET_LABELS, rotation=25, ha="right")
plt.tight_layout(); plt.show()

total = counts.sum()
print("Class counts and percentages:")
for cls, n in counts.items():
    print(f"  {cls:<30s}: {n:4d} rows  ({100*n/total:.1f}%)")

**Reading this chart:**

The bars are almost perfectly equal (~300 rows per class). This is NOT how obesity is
distributed in the real world — in reality, Normal_Weight is the most common and
Obesity_Type_III is rare.

**Why are the classes equal here?**
The dataset was artificially balanced using **SMOTE** (Synthetic Minority Over-sampling Technique).
- The original survey had **498 real respondents** with very unequal class sizes.
- SMOTE generated **1,613 synthetic (fake) respondents** to balance the classes.
- In total: **77% of the data is synthetic**.

**Why balance at all?**
If you have 400 Normal_Weight but only 30 Obesity_Type_III, a model that predicts
"Normal_Weight" for everyone would be 80% accurate but completely useless for Obesity_Type_III.
Balancing forces the model to learn all 7 classes equally well.

**What does this mean for you?**
Model accuracy of 65–78% on this balanced dataset does NOT mean it would achieve
the same in a real unbalanced clinical population. It means the model learned
to distinguish all 7 classes from behavioral data — the scientifically interesting task.

---
## Chapter 3 — The SMOTE Artifact: Why Raw Numbers Are Misleading

### What SMOTE does to ordinal columns

Five columns are **survey responses with fixed integer values**:
- `FCVC`: How often do you eat vegetables? **1** = Never, **2** = Sometimes, **3** = Always
- `NCP`: How many main meals per day? **1**, **2**, **3**, or **4**
- `CH2O`: How much water do you drink? **1** = <1L, **2** = 1–2L, **3** = >2L
- `FAF`: How many days per week do you exercise? **0**, **1**, **2**, or **3**
- `TUE`: How many hours per day on screens? **0**, **1**, or **2**

**SMOTE generates synthetic rows by interpolating between real people.**
Example calculation:
> Real person A: FCVC = 2 (Sometimes vegetables)
> Real person B: FCVC = 3 (Always vegetables)
> SMOTE creates synthetic person C: FCVC = 2 + 0.784 × (3−2) = **2.784**

But 2.784 is not a valid survey answer! Nobody said "I eat vegetables 2.784 times."
If we leave it as 2.784, the model treats it as a meaningful real number — adding noise.

**The chart below shows the problem:**
The red dashed lines mark the ONLY valid answers. Everything between the red lines is synthetic.

In [ ]:
smote_valid = {"FCVC":[1,2,3],"NCP":[1,2,3,4],"CH2O":[1,2,3],"FAF":[0,1,2,3],"TUE":[0,1,2]}
fig, axes = plt.subplots(1,5, figsize=(18,3.5))
for ax, col in zip(axes, smote_valid):
    ax.hist(df[col], bins=60, color="steelblue", edgecolor="white", alpha=0.85)
    for t in smote_valid[col]:
        ax.axvline(t, color="crimson", linewidth=1.5, linestyle="--")
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_xlabel("Raw value from CSV")
plt.suptitle("SMOTE Artifact: Values Spread Between Valid Survey Ticks (red lines = valid answers)",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

print("Examples of SMOTE-interpolated values (not valid survey answers):")
for col in smote_valid:
    floats = df[df[col] != df[col].round()][col].values[:3]
    ints   = sorted(df[col].round().unique())
    print(f"  {col:<6s}: raw examples = {floats}  →  valid answers should be {ints}")

**How to read this chart:**

Each histogram shows how a column's values are distributed. Look at the `FCVC` histogram for example:
- If this were clean data, you'd see **3 tall spikes** at exactly 1, 2, and 3 — nothing else.
- Instead, you see values spread **continuously** from 1.0 to 3.0. The red lines at 1, 2, 3
  show where the real answers should be. All the bars BETWEEN the red lines are SMOTE artefacts.

**What we do about it (Chapter 5):**
Round each value to the nearest valid integer tick and clip to the valid range:
> FCVC = 2.784 → round → **3** ("Always vegetables")
> FAF = 0.23 → round → **0** ("Never exercises")
> TUE = 1.91 → round → **2** (">2 hours/day on screens")

This recovers the original survey answer from the synthetic float. It is lossless for the
498 real respondents (they already had integers) and corrects the 1,613 synthetic rows.

---
## Chapter 4 — Individual Feature Analysis

**What we do in this chapter:**
Analyze every behavioral feature one by one.
For each feature we answer:
- **What** does this feature measure?
- **Why** would it relate to obesity?
- **Example**: Person A vs Person B to make it concrete
- **Chart**: visualize the distribution across obesity classes
- **Read the chart**: exact instructions on how to interpret every element
- **Key number**: Spearman rho — a correlation statistic

**About Spearman correlation (ρ):**
Spearman rho measures how consistently one variable increases as another increases.
- ρ = +1.0 → perfect: as the feature goes up, obesity class always goes up
- ρ = −1.0 → perfect inverse: as feature goes up, obesity class always goes down
- ρ = 0.0 → no relationship
- |ρ| > 0.30 → strong signal for a behavioral feature
- |ρ| 0.10–0.30 → moderate signal
- |ρ| < 0.10 → weak, may be noise

We use **Spearman** (not Pearson) because our obesity classes are *ordered categories*,
not equal-interval numbers — the gap between class 0 and 1 is not necessarily the same
as between class 5 and 6.

In [ ]:
y_num = df[TARGET].map(TARGET_ORDER)  # Convert class names to 0–6 integers for correlation

### Feature 1 — Physical Activity Frequency (FAF)

**What it measures:** How many days per week the respondent is physically active.
Survey values: **0** = none, **1** = 1–2 days/week, **2** = 2–4 days/week, **3** = 4–5 days/week

**Why it would predict obesity:**
Exercise burns calories, builds muscle (which burns more calories at rest), reduces insulin
resistance, and lowers cortisol. It is the strongest *modifiable* factor in energy balance.

**Example:**
> **Person A (FAF = 0):** No structured physical activity. Takes the elevator, drives everywhere.
> **Person B (FAF = 3):** Exercises 5 days per week — gym, running, or sport.
> All else equal, Person B burns ~2,000 more kcal per week from exercise alone.
> Over one year: ~104,000 kcal difference ≈ 13 kg of body fat.

**What to look for in the boxplot:**
Each box covers the middle 50% of values for that obesity class (25th to 75th percentile).
The line in the middle of the box is the **median** (the typical value for that class).
→ If FAF protects against obesity, the medians should **decrease** from left to right.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,5))

# Left: FAF by class
sns.boxplot(data=df, x=TARGET, y="FAF", order=TARGET_LABELS, palette=PALETTE, ax=axes[0], linewidth=0.8)
axes[0].set_xticklabels(TARGET_LABELS, rotation=25, ha="right")
axes[0].set_title("Physical Activity (FAF) by Obesity Class", fontsize=11, fontweight="bold")
axes[0].set_ylabel("FAF  (0=no exercise, 3=4–5 days/week)")
axes[0].set_xlabel("")

# Right: TUE by class
sns.boxplot(data=df, x=TARGET, y="TUE", order=TARGET_LABELS, palette=PALETTE, ax=axes[1], linewidth=0.8)
axes[1].set_xticklabels(TARGET_LABELS, rotation=25, ha="right")
axes[1].set_title("Screen / Technology Use (TUE) by Obesity Class", fontsize=11, fontweight="bold")
axes[1].set_ylabel("TUE  (0=<1h, 1=1–3h, 2=>3h per day)")
axes[1].set_xlabel("")

plt.tight_layout(); plt.show()

rho_faf, p_faf = stats.spearmanr(df["FAF"], y_num)
rho_tue, p_tue = stats.spearmanr(df["TUE"], y_num)
print(f"FAF Spearman ρ = {rho_faf:+.3f}  (p={p_faf:.2e})")
print(f"TUE Spearman ρ = {rho_tue:+.3f}  (p={p_tue:.2e})")

print("\nMedian FAF by class:")
for cls in TARGET_LABELS:
    med = df[df[TARGET]==cls]["FAF"].median()
    bar = "█"*int(med*6)
    print(f"  {cls:<30s}: median={med:.2f}  {bar}")

print("\nMedian TUE by class:")
for cls in TARGET_LABELS:
    med = df[df[TARGET]==cls]["TUE"].median()
    bar = "█"*int(med*6)
    print(f"  {cls:<30s}: median={med:.2f}  {bar}")

**Reading the FAF chart (LEFT) — Physical Activity:**

The printed "Median FAF by class" table shows the actual medians from the data:

| Class | Median FAF |
|-------|-----------|
| Insufficient_Weight | 1.34 |
| Normal_Weight | 1.00 |
| Overweight_I | 1.00 |
| Overweight_II | 0.96 |
| Obesity_Type_I | 0.99 |
| Obesity_Type_II | 0.99 |
| **Obesity_Type_III** | **0.22** |

**What this actually means:** Five of the seven classes sit at almost the same median (~1.0 days/week of exercise). There is NO smooth protective trend across classes. **Only Obesity_Type_III (severely obese) drops dramatically to 0.22** — meaning the most severely obese people exercise almost not at all.

**FAF Spearman ρ = −0.180.** Negative (more exercise → lower class number is correct), but the magnitude is only moderate, not strong, because the separation only appears at the extreme end (OB_III), not across the whole spectrum.

---

**Reading the TUE chart (RIGHT) — Screen Time:**

**TUE Spearman ρ = −0.076 (near zero).** This is a weak, almost meaningless number.

The boxplot medians are **chaotic** — they do not increase or decrease in any consistent direction across obesity classes. High screen time does NOT reliably predict obesity class in this dataset.

**Why include TUE then?** It becomes meaningful only when *combined* with FAF as `screen_activity` in Feature Engineering (Chapter 6). A person who BOTH exercises very little AND spends 4+ hours on screens has a very different risk profile from someone who just watches TV but is otherwise active.


### Feature 2 — Eating Between Meals (CAEC) and High-Calorie Food (FAVC)

**What CAEC measures:** How often does the respondent eat between main meals (snacking)?
- **no** (0) = never snacks
- **Sometimes** (1) = occasional snacks
- **Frequently** (2) = regular snacking
- **Always** (3) = constant snacking throughout the day

**What FAVC measures:** Does the respondent frequently eat high-calorie food? (yes/no)

**Example:**
> **Person A (CAEC=no, FAVC=no):** Three square meals, no junk food. Controlled intake.
> **Person B (CAEC=Always, FAVC=yes):** Snacks constantly and the snacks are chips, chocolate,
> fast food. Extra 400–600 kcal/day from snacking alone → ~25 kg of fat per year if sustained.

**LEFT chart — Stacked bar:**
Each bar = one snacking frequency group. The coloured segments = obesity class shares.
*How to read it:* Look at how much of each bar is made of dark green/yellow (obesity colours)
vs blue/purple (normal/underweight colours). More dark = more obesity in that group.

**RIGHT chart — Heatmap:**
This shows the OVERLAP between CAEC and FAVC.
- Each row = one snacking frequency level
- Each column = FAVC yes/no
- Each cell value = **fraction** of that row group who also chose that FAVC answer
- Example: if cell (Always, yes) = **0.89**, it means **89% of "Always snackers" also
  frequently eat high-calorie food**. Only 11% snack constantly but maintain clean food quality.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,5))
caec_ord = ["no","Sometimes","Frequently","Always"]

# Left: CAEC stacked bar
ct = (df.groupby(["CAEC",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0))
ct_norm = ct.div(ct.sum(axis=1),axis=0)
ct_norm.reindex(caec_ord).plot(kind="bar",stacked=True,ax=axes[0],colormap="viridis",
    edgecolor="white",linewidth=0.3,legend=False)
axes[0].set_title("Snacking Frequency (CAEC) — % in each obesity class", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Snacking Frequency")
axes[0].tick_params(axis="x",rotation=15)
for i,row in enumerate(caec_ord):
    n = len(df[df["CAEC"]==row])
    axes[0].text(i, 1.02, f"n={n}", ha="center", fontsize=8, color="gray")

# Right: CAEC × FAVC heatmap
cross = pd.crosstab(df["CAEC"],df["FAVC"])
cross_pct = cross.div(cross.sum(axis=1),axis=0)
sns.heatmap(cross_pct.reindex(caec_ord), annot=True, fmt=".2f",
            cmap="YlOrRd", linewidths=0.4, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("CAEC × FAVC Heatmap
(each cell = fraction of that row's group)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("FAVC: frequent high-calorie food? (no / yes)")
axes[1].set_ylabel("CAEC: snacking frequency")

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles,labels_,loc="lower center",ncol=4,fontsize=7,title="Obesity Level",bbox_to_anchor=(0.5,-0.08))
plt.tight_layout(); plt.show()

rho_caec, p_caec = stats.spearmanr(df["CAEC"].map(FREQ_MAP), y_num)
rho_favc, p_favc = stats.spearmanr((df["FAVC"]=="yes").astype(int), y_num)
print(f"CAEC Spearman ρ = {rho_caec:+.3f}  (p={p_caec:.2e})")
print(f"FAVC Spearman ρ = {rho_favc:+.3f}  (p={p_favc:.2e})")
print()
print("Obesity share (%) by snacking level:")
for g in caec_ord:
    sub = df[df["CAEC"]==g]
    pct_obese = sub[TARGET].str.startswith("Obesity").mean()*100
    pct_normal = (sub[TARGET]=="Normal_Weight").mean()*100
    print(f"  CAEC={g:<12s}: {pct_obese:.1f}% obese,  {pct_normal:.1f}% Normal_Weight  (n={len(sub)})")

**Reading the CAEC stacked bar — the SMOTE artifact in action:**

The printed obesity rates per CAEC level come directly from the data:

| CAEC level | n (sample size) | % obese |
|-----------|----------------|---------|
| no | 51 | **3.9%** |
| Sometimes | 1,765 | **54.1%** |
| Frequently | 242 | **3.3%** |
| Always | 53 | **15.1%** |

**This is counter-intuitive and backwards from what we would expect clinically.**
"Frequently" should have MORE obesity than "Sometimes", but the data shows "Frequently" at only 3.3% while "Sometimes" at 54.1%.

**Why does this happen? SMOTE distortion.**
The original dataset was tiny. SMOTE generated synthetic rows by interpolating between real neighbours. "Sometimes" was the most common CAEC value in the original data — so SMOTE created the vast majority of its 1,765 synthetic rows as "Sometimes", including most of the synthetic obese rows. "Frequently" and "Always" have tiny counts (242 and 53) because SMOTE created far fewer rows there.

**Spearman ρ = −0.353.** The negative sign is also backwards from clinical expectation — it says eating between meals LESS often predicts obesity. This is purely a SMOTE artifact, not biological reality.

---

**Reading the CAEC×FAVC heatmap — proportion of "FAVC=yes" within each CAEC group:**

Each cell shows what fraction of people in that CAEC group ALSO eat high-calorie food (FAVC=yes):

| CAEC | FAVC=no | FAVC=yes |
|------|--------|---------|
| no | 0.18 | **0.82** |
| Sometimes | 0.09 | **0.91** |
| Frequently | 0.28 | **0.72** |
| Always | 0.23 | **0.77** |

**Key insight:** Across ALL snacking levels, 72–91% of people also eat high-calorie food. Bad eating habits cluster together. "Sometimes" snackers have the highest rate of also eating junk food (91%). This clustering justifies combining CAEC and FAVC in a single dietary risk composite (Chapter 6).


### Feature 3 — Vegetable Consumption (FCVC)

**What it measures:** How often vegetables appear in meals.
**1** = Never  |  **2** = Sometimes  |  **3** = Always

**Example:**
> **Person A (FCVC=1):** Meals consist of rice, meat, bread — zero vegetables.
> **Person B (FCVC=3):** Every plate has two vegetable sides alongside the protein.
> Dietary fibre (abundant in vegetables) slows glucose absorption, extends satiety for 2–3
> extra hours, and feeds gut bacteria that regulate appetite hormones. Person A's blood sugar
> spikes higher and faster, triggering more insulin, storing more fat, and making them hungry
> again sooner.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,5))
sns.boxplot(data=df,x=TARGET,y="FCVC",order=TARGET_LABELS,palette=PALETTE,ax=axes[0],linewidth=0.8)
axes[0].set_xticklabels(TARGET_LABELS,rotation=25,ha="right")
axes[0].set_title("Vegetable Frequency (FCVC) by Obesity Class",fontsize=11,fontweight="bold")
axes[0].set_ylabel("FCVC  (1=Never, 2=Sometimes, 3=Always)")
axes[0].set_xlabel("")

df["_fcvc"] = np.clip(df["FCVC"],1,3).round().astype(int)
ct = df.groupby(["_fcvc",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct.div(ct.sum(axis=1),axis=0).plot(kind="bar",stacked=True,ax=axes[1],colormap="viridis",
    edgecolor="white",linewidth=0.3,legend=False)
axes[1].set_xlabel("FCVC  (1/2/3)"); axes[1].tick_params(axis="x",rotation=0)
axes[1].set_title("FCVC Level — % in each obesity class",fontsize=11,fontweight="bold")
handles,labels_=axes[1].get_legend_handles_labels()
fig.legend(handles,labels_,loc="lower center",ncol=4,fontsize=7,title="Obesity Level",bbox_to_anchor=(0.5,-0.08))
plt.tight_layout(); plt.show()

rho,p = stats.spearmanr(df["FCVC"],y_num)
print(f"FCVC Spearman ρ = {rho:+.3f}  (p={p:.2e})")
print()
for lvl in [1,2,3]:
    sub=df[df["_fcvc"]==lvl]; lbl={1:"Never  ",2:"Sometimes",3:"Always "}[lvl]
    print(f"  FCVC={lvl} ({lbl}): {sub[TARGET].str.startswith('Obesity').mean()*100:.1f}% obese  (n={len(sub)})")
df.drop(columns="_fcvc",inplace=True)

**Reading the FCVC charts — another SMOTE reversal:**

The printed obesity rates per FCVC level:

| FCVC level | n | % obese |
|-----------|---|---------|
| 1 — Never | 102 | 37.3% |
| 2 — Sometimes | 1,013 | 38.9% |
| 3 — Always | 996 | **54.2%** |

**This is the opposite of what nutrition science says.** Eating more vegetables should be protective, yet the data shows FCVC=3 (Always eats vegetables) has the HIGHEST obesity rate at 54.2%.

**Spearman ρ = +0.260.** Positive means "more vegetables → higher obesity class" — a completely reversed signal.

**Why? SMOTE artifact — again.**
FCVC=3 has 996 rows, meaning SMOTE created hundreds of synthetic obese people who were assigned FCVC=3 because their obese neighbours happened to report eating lots of vegetables. The synthetic interpolation did not respect the biological direction of the relationship — it just blended numeric features. This creates a spurious "vegetables predict obesity" signal that is entirely an artifact of synthetic data generation.

**Bottom line:** Do NOT interpret FCVC=+0.260 as "vegetables cause obesity." It tells you that SMOTE corrupted this feature's relationship with the target. We flag it, keep it in the model (tree models can still extract partial signal), and rely on other features for the main clinical interpretation.


### Feature 4 — Number of Main Meals (NCP)

**What it measures:** How many main meals per day. Values: **1**, **2**, **3**, or **4**.

**Example:**
> **Person A (NCP=1):** Skips breakfast and lunch, eats one huge dinner at 9 pm.
> **Person B (NCP=3):** Breakfast, lunch, dinner — evenly spaced.
>
> Person A's large evening meal causes a massive insulin spike when the body is winding
> down for sleep. The excess glucose is efficiently stored as fat rather than burned.
> Person B's three smaller meals keep blood sugar stable all day.
> However — NCP=4 (four meals/day) could mean Person B eats FOUR LARGE meals,
> not four small ones. The survey does not record portion size.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,5))
sns.boxplot(data=df,x=TARGET,y="NCP",order=TARGET_LABELS,palette=PALETTE,ax=axes[0],linewidth=0.8)
axes[0].set_xticklabels(TARGET_LABELS,rotation=25,ha="right")
axes[0].set_title("Main Meal Frequency (NCP) by Obesity Class",fontsize=11,fontweight="bold")
axes[0].set_ylabel("NCP  (1–4 meals/day)")
axes[0].set_xlabel("")

df["_ncp"] = np.clip(df["NCP"],1,4).round().astype(int)
ct = df.groupby(["_ncp",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct.div(ct.sum(axis=1),axis=0).plot(kind="bar",stacked=True,ax=axes[1],colormap="viridis",
    edgecolor="white",linewidth=0.3,legend=False)
axes[1].set_xlabel("NCP  (meals/day)"); axes[1].tick_params(axis="x",rotation=0)
axes[1].set_title("NCP Level — % in each obesity class",fontsize=11,fontweight="bold")
handles,labels_=axes[1].get_legend_handles_labels()
fig.legend(handles,labels_,loc="lower center",ncol=4,fontsize=7,title="Obesity Level",bbox_to_anchor=(0.5,-0.08))
plt.tight_layout(); plt.show()

rho,p = stats.spearmanr(df["NCP"],y_num)
print(f"NCP Spearman ρ = {rho:+.3f}  (p={p:.2e})")
print()
for n in sorted(df["_ncp"].unique()):
    sub=df[df["_ncp"]==n]
    print(f"  NCP={n} meals/day: {sub[TARGET].str.startswith('Obesity').mean()*100:.1f}% obese  (n={len(sub)})")
df.drop(columns="_ncp",inplace=True)

**Reading the NCP charts — weak and confused signal:**

The printed obesity rates per NCP level:

| NCP | n | % obese |
|-----|---|---------|
| 1 meal/day | 316 | 32.6% |
| 2 meals/day | 176 | 47.2% |
| 3 meals/day | 1,470 | **52.9%** |
| 4 meals/day | 149 | 6.0% |

**Spearman ρ = −0.053 (near zero).** NCP has essentially no predictive value as a standalone feature.

**What the bar chart actually shows:**
- NCP=3 (three meals/day) dominates with 1,470 rows — it is by far the most common answer at every obesity level. This creates a "ceiling" where every class looks similar.
- NCP=4 appears to be protective (6% obese) but this may reflect health-conscious people who plan structured frequent small meals, not people who eat more overall.
- NCP=1 at 32.6% might seem lower, but n=316 and these could be people who skip meals for weight management reasons (reverse causality).

**Conclusion:** Meal frequency alone does not predict obesity in this dataset. Three meals is the universal norm regardless of weight class. We still include NCP in the `caloric_risk` composite to capture any residual joint signal with snacking and food quality.


### Feature 5 — Water Intake (CH2O)

**Values:** **1** = less than 1 litre/day  |  **2** = 1–2 litres/day  |  **3** = more than 2 litres/day

**Example:**
> **Person A (CH2O=1):** Drinks cola, coffee, juice — barely 500 mL water/day.
> **Person B (CH2O=3):** Carries a 2-litre bottle, finishes it every day.
>
> **Direct effect:** Water = zero calories. If Person A replaces 2 cans of cola with water,
> they eliminate 280 kcal/day = 10 kg of fat per year in pure caloric math.
>
> **Indirect effect:** The brain's thirst signal and hunger signal travel the same neural path.
> When Person A is mildly dehydrated, the brain sends a "get energy" signal that is often
> misread as hunger. Person A eats extra food when they actually needed water.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,5))
sns.boxplot(data=df,x=TARGET,y="CH2O",order=TARGET_LABELS,palette=PALETTE,ax=axes[0],linewidth=0.8)
axes[0].set_xticklabels(TARGET_LABELS,rotation=25,ha="right")
axes[0].set_title("Daily Water Intake (CH2O) by Obesity Class",fontsize=11,fontweight="bold")
axes[0].set_ylabel("CH2O  (1=<1L, 2=1–2L, 3=>2L)")
axes[0].set_xlabel("")

df["_ch2o"] = np.clip(df["CH2O"],1,3).round().astype(int)
ct = df.groupby(["_ch2o",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct.div(ct.sum(axis=1),axis=0).plot(kind="bar",stacked=True,ax=axes[1],colormap="viridis",
    edgecolor="white",linewidth=0.3,legend=False)
axes[1].set_xlabel("CH2O level  (1/2/3)"); axes[1].tick_params(axis="x",rotation=0)
axes[1].set_title("CH2O Level — % in each obesity class",fontsize=11,fontweight="bold")
handles,labels_=axes[1].get_legend_handles_labels()
fig.legend(handles,labels_,loc="lower center",ncol=4,fontsize=7,title="Obesity Level",bbox_to_anchor=(0.5,-0.08))
plt.tight_layout(); plt.show()

rho,p = stats.spearmanr(df["CH2O"],y_num)
print(f"CH2O Spearman ρ = {rho:+.3f}  (p={p:.2e})")
for lvl,lbl in [(1,"<1 L   "),(2,"1–2 L  "),(3,">2 L   ")]:
    sub=df[df["_ch2o"]==lvl]
    print(f"  CH2O={lvl} ({lbl}): {sub[TARGET].str.startswith('Obesity').mean()*100:.1f}% obese  (n={len(sub)})")
df.drop(columns="_ch2o",inplace=True)

**Reading the CH2O charts — reversed by SMOTE and physiology:**

The printed obesity rates per CH2O level:

| CH2O | n | % obese |
|------|---|---------|
| 1 — Less than 1L/day | 485 | 43.5% |
| 2 — 1–2L/day | 1,110 | 41.8% |
| 3 — More than 2L/day | 516 | **57.6%** |

**Spearman ρ = +0.150.** Positive means "more water → higher obesity class" — again reversed from clinical expectation.

**Why does CH2O=3 have the HIGHEST obesity rate?**
Two mechanisms compound here:

1. **Reverse causality / physiology:** A heavier body has higher metabolic water needs. A person who weighs 120 kg genuinely needs and drinks more water than a person who weighs 60 kg. So obesity causes high water intake, not the other way around. The survey captures a snapshot — it doesn't tell us which came first.

2. **Survey conflation:** The original survey question may have asked about ALL beverages (water, juice, soda, coffee), not strictly water. If obese respondents drink large amounts of sugary drinks and counted those as "water", the high CH2O=3 obesity rate would be expected.

3. **SMOTE:** Same as FCVC — CH2O=2 and CH2O=3 are the most common values, so SMOTE created many synthetic obese rows there.

**Bottom line:** Do not interpret "drink more water" as a risk factor. The ρ=+0.150 is a SMOTE artifact compounded by reverse causality. We keep CH2O in the model because the tree can still extract some pattern, but we do not report it as a risk factor.


### Features 6–9 — Binary Risk Behaviors

Four binary features (yes/no):
- **SMOKE**: Does the respondent smoke?
- **SCC**: Does the respondent monitor their calorie intake?
- **FAVC**: Does the respondent frequently eat high-calorie food?
- **family_history_with_overweight**: Does a family member have overweight?

**Why SMOKE is counter-intuitive:**
Nicotine raises basal metabolic rate ~10% and suppresses appetite.
Cross-sectional data (one snapshot in time) shows smokers weigh less than non-smokers.
This does NOT mean smoking prevents obesity — when people quit, weight often rebounds +4 kg.
Additionally only ~2% of this dataset smokes, giving very low statistical power.
**Expect: SMOKE ρ ≈ 0, near-zero predictive value.**

**Why SCC shows reverse causality:**
People who track calories are usually *already concerned* about their weight.
The tracking is a *response* to weight gain, not a prevention.
**Expect: SCC ρ ≈ 0 or slightly positive** (calorie counters span all obesity levels).

**Why family_history is the strongest single predictor:**
Genetics control baseline metabolism, fat-storage hormone levels, appetite regulation,
and fat distribution patterns. A person with obese parents has an estimated 40–70% higher
risk from genes alone, before any lifestyle factor.

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(16,10))
bin_features = [("SMOKE","Smoking"),("SCC","Calorie Monitoring"),
                ("FAVC","High-Calorie Food"),("family_history_with_overweight","Family History")]

for ax, (col,label) in zip(axes.flat, bin_features):
    ct = df.groupby([col,TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
    ct_norm = ct.div(ct.sum(axis=1),axis=0)
    ct_norm.plot(kind="bar",stacked=True,ax=ax,colormap="viridis",edgecolor="white",linewidth=0.3,legend=False)
    ax.set_title(f"{label} ({col})",fontsize=11,fontweight="bold")
    ax.set_xlabel(col); ax.tick_params(axis="x",rotation=0)
    counts = df[col].value_counts()
    for i,(val,_) in enumerate(ct_norm.iterrows()):
        ax.text(i, 1.02, f"n={counts.get(val,0)}", ha="center", fontsize=8, color="gray")

handles,labels_=axes[0,0].get_legend_handles_labels()
fig.legend(handles,labels_,loc="lower center",ncol=4,fontsize=8,title="Obesity Level",bbox_to_anchor=(0.5,-0.02))
plt.suptitle("Binary Feature Distributions — % in each obesity class",fontsize=13,fontweight="bold")
plt.tight_layout(); plt.show()

print(f"{'Feature':<42} {'ρ':>8}  {'p-value':>12}  Interpretation")
print("─"*80)
for col,label in bin_features:
    x = (df[col]=="yes").astype(int)
    rho,p = stats.spearmanr(x,y_num)
    interp = ("STRONG risk" if rho>0.3 else "Moderate risk" if rho>0.1
              else "Strong protect" if rho<-0.3 else "Near zero / ambiguous")
    print(f"  {label:<40} {rho:+.3f}  p={p:.2e}  {interp}")

**Reading the binary feature charts — actual data results:**

**SMOKE (Spearman ρ = +0.003 — essentially zero):**
"Yes" smokers: ~50% obese (n=44 only!). "No" non-smokers: ~46% obese.
The bars look almost identical and the sample of smokers is tiny (44 people = 2% of dataset). No reliable signal.

---

**SCC — Calorie Monitoring (Spearman ρ = −0.194):**

Actual obesity rates:
| SCC | n | % obese |
|-----|---|---------|
| yes (monitors calories) | 96 | **3.1%** |
| no (does not monitor) | ~2,015 | **48.1%** |

**The "yes" bar is almost entirely NON-obese (96.9% of calorie trackers are NOT obese).**
This is NOT reverse causality (the template explanation was wrong). Calorie monitoring in this dataset is predominantly a behaviour of health-conscious, thin people. They track calories BECAUSE they care about staying lean — not because they gained weight. This is a very strong protective signal (ρ = −0.194).

---

**FAVC — High-Calorie Food (Spearman ρ = +0.250):**

| FAVC | n | % obese |
|------|---|---------|
| yes | 1,866 (88% of data) | **51.1%** |
| no | 245 | **7.8%** |

Clear risk signal. Over half of high-calorie-food eaters are obese; almost none of the non-eaters are. The n= imbalance (1866 vs 245) means the "yes" group dominates the training data.

---

**Family History (Spearman ρ = +0.500 — the SINGLE STRONGEST predictor):**

| family_history | n | % obese |
|---------------|---|---------|
| yes | 1,726 | **55.9%** |
| no | 385 | **2.1%** |

**A 54-percentage-point gap.** This is extraordinary. Among people with family history of overweight, 56% are obese. Among those without it, only 2% are. No other single feature comes close to this separation. Genetic predisposition dwarfs every lifestyle factor in this dataset.


### Feature 10 — Alcohol Consumption (CALC)

**Values:** **no**  |  **Sometimes**  |  **Frequently**  |  **Always**

**Example:**
> **Person A (CALC=no):** Drinks no alcohol.
> **Person B (CALC=Frequently):** Two glasses of wine 5 nights/week.
>
> 2 glasses wine ≈ 300 kcal. Five nights = **1,500 extra kcal/week**.
> Over 52 weeks = 78,000 kcal ≈ **10 kg of extra fat per year** (pure caloric calculation).
> Beyond calories: alcohol disrupts deep sleep (growth hormone secreted during deep sleep
> controls fat metabolism) and impairs next-morning food choices via cortisol elevation.

**Nuance — the "sick quitter" effect:**
Some people who say "no" to alcohol stopped drinking because of illness or medication —
making the "no" group appear *less healthy* than light drinkers. This is a known
epidemiological artefact in cross-sectional surveys.

In [ ]:
calc_order = ["no","Sometimes","Frequently","Always"]
fig, axes = plt.subplots(1,2, figsize=(16,5))

ct = df.groupby(["CALC",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct_norm = ct.div(ct.sum(axis=1),axis=0)
ct_norm.reindex(calc_order).plot(kind="bar",stacked=True,ax=axes[0],colormap="viridis",
    edgecolor="white",linewidth=0.3,legend=False)
axes[0].set_title("Alcohol Frequency (CALC) — % in each obesity class",fontsize=11,fontweight="bold")
axes[0].tick_params(axis="x",rotation=15)

rate = df.groupby("CALC").apply(lambda g: g[TARGET].str.startswith("Obesity").mean()*100).reindex(calc_order)
bars = axes[1].bar(calc_order, rate.values, color=sns.color_palette("OrRd",4), edgecolor="white")
axes[1].bar_label(bars, fmt="%.1f%%", padding=3)
axes[1].set_title("Obesity Rate by Alcohol Frequency",fontsize=11,fontweight="bold")
axes[1].set_ylabel("% of group who are obese"); axes[1].tick_params(axis="x",rotation=15)
axes[1].set_ylim(0, rate.max()*1.25)

handles,labels_=axes[0].get_legend_handles_labels()
fig.legend(handles,labels_,loc="lower center",ncol=4,fontsize=7,title="Obesity Level",bbox_to_anchor=(0.5,-0.08))
plt.tight_layout(); plt.show()

calc_num = df["CALC"].map({"no":0,"Sometimes":1,"Frequently":2,"Always":3})
rho,p = stats.spearmanr(calc_num,y_num)
print(f"CALC Spearman ρ = {rho:+.3f}  (p={p:.2e})")
print("\nObesity rate and group size:")
for lvl in calc_order:
    sub=df[df["CALC"]==lvl]
    print(f"  {lvl:<12s}: {sub[TARGET].str.startswith('Obesity').mean()*100:.1f}% obese  (n={len(sub)})")

**Reading the CALC charts — alcohol consumption:**

Actual obesity rates per CALC level:

| CALC | n | % obese |
|------|---|---------|
| no | 639 | 37.1% |
| Sometimes | 1,401 | **51.3%** |
| Frequently | 70 | 22.9% |
| **Always** | **1** | **0%** |

**Spearman ρ = +0.168.** Positive (more alcohol → higher obesity) but the pattern is not monotone.

**Critical point: CALC=Always has n=1 (ONE person).** This is statistically meaningless. Any percentage from this group is noise, not a finding. The chart will show a bar for "Always" but treat it as an artifact.

**What the data actually shows:**
"Sometimes" (n=1,401, the dominant group) has 51.3% obesity vs "no" at 37.1%. This is the meaningful comparison — occasional drinking correlates with +14 percentage points of obesity risk. "Frequently" appears lower (22.9%) but n=70 is still small; this may be another SMOTE concentration effect.

**Conclusion:** Occasional alcohol consumption (Sometimes) is associated with meaningfully higher obesity rates. Always-drinkers cannot be interpreted — the single observation is a data curiosity, not evidence.


### Features 11–12 — Age and Gender

**Age (continuous, range 14–61):**

**Example:**
> **Person A, age 17:** Growing, high metabolic rate (~2,800 kcal/day at rest if active).
> School sport provides built-in exercise. High insulin sensitivity — calories are used efficiently.
>
> **Person B, age 45:** Basal metabolic rate declined ~5% per decade since age 25 (~−10% total).
> Desk job. Testosterone (men) or oestrogen (women) declining → fat redistribution accelerates.
> Muscle mass declining (sarcopenia) → even less resting calorie burn.
> 20 years of accumulated dietary habits.

**Gender:**
- Men: more muscle mass → higher resting metabolic rate
- Women: oestrogen promotes subcutaneous fat (hips/thighs) which is metabolically safer
- Men's visceral (belly) fat carries higher cardiovascular risk per kg
- Post-40: oestrogen decline in women accelerates abdominal fat accumulation

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(16,10))

# Age boxplot
sns.boxplot(data=df,x=TARGET,y="Age",order=TARGET_LABELS,palette=PALETTE,ax=axes[0,0],linewidth=0.8)
axes[0,0].set_xticklabels(TARGET_LABELS,rotation=25,ha="right")
axes[0,0].set_title("Age by Obesity Class",fontsize=11,fontweight="bold")
axes[0,0].set_ylabel("Age (years)"); axes[0,0].set_xlabel("")

# Age group stacked bar
df["_age_grp"] = pd.cut(df["Age"],bins=[0,18,25,35,50,99],
    labels=["Teen(≤18)","Young(19-25)","Adult(26-35)","Mid-Age(36-50)","Senior(51+)"],include_lowest=True)
ct = df.groupby(["_age_grp",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct.div(ct.sum(axis=1),axis=0).plot(kind="bar",stacked=True,ax=axes[0,1],colormap="viridis",
    edgecolor="white",linewidth=0.3)
axes[0,1].set_title("Age Group — % in each obesity class",fontsize=11,fontweight="bold")
axes[0,1].tick_params(axis="x",rotation=15)
axes[0,1].legend(fontsize=7,bbox_to_anchor=(1.01,1),loc="upper left",title="Obesity Level")

# Gender stacked bar
ct_g = df.groupby(["Gender",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct_g.div(ct_g.sum(axis=1),axis=0).plot(kind="bar",stacked=True,ax=axes[1,0],colormap="viridis",
    edgecolor="white",linewidth=0.3,legend=False)
axes[1,0].set_title("Gender — % in each obesity class",fontsize=11,fontweight="bold")
axes[1,0].tick_params(axis="x",rotation=0)

# Gender within each class
ct_g2 = df.groupby([TARGET,"Gender"]).size().unstack("Gender").reindex(TARGET_LABELS,fill_value=0)
ct_g2.div(ct_g2.sum(axis=1),axis=0).plot(kind="bar",ax=axes[1,1],
    color=["#E07A5F","#3D405B"],edgecolor="white",linewidth=0.6)
axes[1,1].set_title("Male/Female split WITHIN each obesity class",fontsize=11,fontweight="bold")
axes[1,1].tick_params(axis="x",rotation=25); axes[1,1].set_ylabel("Proportion")
axes[1,1].legend(fontsize=9)

plt.tight_layout(); plt.show()

rho_age,p_age = stats.spearmanr(df["Age"],y_num)
rho_gen,p_gen = stats.spearmanr((df["Gender"]=="Male").astype(int),y_num)
print(f"Age    ρ = {rho_age:+.3f}  (p={p_age:.2e})")
print(f"Gender ρ = {rho_gen:+.3f}  (p={p_gen:.2e})  [Male=1]")
print()
print("Mean age by class:")
for cls in TARGET_LABELS:
    m = df[df[TARGET]==cls]["Age"].mean()
    print(f"  {cls:<30s}: {m:.1f} years")
df.drop(columns="_age_grp",inplace=True)

**Reading the Age charts:**

**TOP-LEFT boxplot — Mean age by obesity class:**

| Class | Mean Age |
|-------|---------|
| Insufficient_Weight | 19.8 years |
| Normal_Weight | 21.7 years |
| Overweight_I | 23.4 years |
| Overweight_II | 27.0 years |
| Obesity_Type_I | 25.9 years |
| Obesity_Type_II | 28.2 years |
| **Obesity_Type_III** | **23.5 years** |

**Spearman ρ = +0.409 (strong).** Age is the second strongest individual predictor after family history. There is a clear upward trend through OW_II (27y) and OB_II (28y).

**Surprise: Obesity_Type_III mean age = 23.5 years — YOUNGER than OW_II (27y) and OB_II (28y).**
This appears as a "dip" at the end of the age progression. It reflects severe EARLY-ONSET obesity in young adults. In the Latin American context of this dataset (ObesityLevelEstimation study), severe obesity striking people in their early 20s is a documented public health pattern — genetic predisposition combined with ultra-processed food environments creates early-life obesity trajectories.

---

**Reading the Gender charts:**

**Spearman ρ = −0.037 (essentially zero).**
Male obesity rate: **46.0%**
Female obesity rate: **46.1%**

These are IDENTICAL. The two bars in the stacked chart will look virtually the same. Gender provides no meaningful predictive signal in this dataset — it does NOT separate obesity classes. We include Gender in the model (as Male/Female dummies) but expect it to appear near the bottom of feature importance rankings.


### Feature 13 — Transportation Mode (MTRANS)

**Values:** Walking  |  Bike  |  Public_Transportation  |  Motorbike  |  Automobile

**Example:**
> **Person A (MTRANS=Walking):** Walks 20 minutes to the train station, 15 minutes to the office.
> 35 minutes of brisk walking × 5 days = **175 minutes of cardio per week** — built in, automatic,
> requires no gym membership, no willpower after a hard day.
>
> **Person B (MTRANS=Automobile):** Door to door by car. Zero incidental exercise from transport.
> Gets exactly the same exercise as their gym schedule — which, on a tired Tuesday, might be zero.

This is **incidental physical activity** — the most sustainable exercise because it is
structurally built into the day, not dependent on motivation.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,5))

ct = df.groupby(["MTRANS",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct_norm = ct.div(ct.sum(axis=1),axis=0)
ct_norm["_obese"] = ct_norm[[c for c in TARGET_LABELS if "Obesity" in c]].sum(axis=1)
ct_sorted = ct_norm.sort_values("_obese").drop(columns="_obese")

ct_sorted.plot(kind="bar",stacked=True,ax=axes[0],colormap="viridis",
    edgecolor="white",linewidth=0.3,legend=False)
axes[0].set_title("Transport Mode — sorted by obesity share (lowest first)",fontsize=11,fontweight="bold")
axes[0].tick_params(axis="x",rotation=20)
for i,mode in enumerate(ct_sorted.index):
    n = len(df[df["MTRANS"]==mode])
    axes[0].text(i, 1.02, f"n={n}", ha="center", fontsize=7, color="gray")

active = df["MTRANS"].isin(["Walking","Bike"]).astype(int)
df["_transport"] = active
ct2 = df.groupby(["_transport",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct2_norm = ct2.div(ct2.sum(axis=1),axis=0)
ct2_norm.index = ["Passive (Car/Moto/Bus)","Active (Walk/Bike)"]
ct2_norm.plot(kind="bar",stacked=True,ax=axes[1],colormap="viridis",edgecolor="white",linewidth=0.3,legend=False)
axes[1].set_title("Active vs Passive Transport — obesity class share",fontsize=11,fontweight="bold")
axes[1].tick_params(axis="x",rotation=15)
handles,labels_=axes[0].get_legend_handles_labels()
fig.legend(handles,labels_,loc="lower center",ncol=4,fontsize=7,title="Obesity Level",bbox_to_anchor=(0.5,-0.08))
plt.tight_layout(); plt.show()

df.drop(columns="_transport",inplace=True)
print("Obesity rate by transport mode:")
for mode in ct_sorted.index:
    sub=df[df["MTRANS"]==mode]
    print(f"  {mode:<25s}: {sub[TARGET].str.startswith('Obesity').mean()*100:.1f}% obese  (n={len(sub)})")

**Reading the MTRANS charts — transportation mode:**

Actual obesity rates per transport mode:

| MTRANS | n | % obese |
|--------|---|---------|
| Walking | 56 | **5.4%** |
| Bike | 7 | 14.3% |
| Motorbike | 11 | 27.3% |
| Public_Transportation | 1,580 | **48.0%** |
| Automobile | 457 | **45.1%** |

**Walking is the most protective** (5.4% obese) — but CRITICAL CAVEAT: n=56 only.
56 people walk to work/school in this dataset. The 5.4% figure is compelling but comes from a tiny sample; it cannot be generalised reliably.

**Bike has n=7** — statistically meaningless. Ignore any % for Bike.

**Public Transportation (n=1,580) vs Automobile (n=457):** Similar rates, 48% vs 45%. The vast majority of people (97% of the dataset) use either public transit or car — these are the practically significant groups, and they have similar obesity rates. The modest 3-point difference between transit and car use is not clinically meaningful.

**RIGHT chart — Active vs Passive transport:**
Active = Walk + Bike (n=63, 2.9% of dataset), Passive = everything else (97%).
The "Active" bar will show a much lower obesity fraction, but this result is driven almost entirely by the 56 walkers. Given the tiny sample, interpret this pattern cautiously.

**Conclusion:** Transport mode has a suggestive signal (walkers are leaner), but the sample sizes are too imbalanced to draw firm conclusions. The model will learn this pattern but should not be expected to generalise well to populations where more people walk.


---
## Chapter 5 — Data Cleaning

### Step 1: Recover integer survey values from SMOTE floats

**What we do:** For each of the five ordinal columns, clip to the valid range and round.

**Formula:** `col_int = clip(col, min_valid, max_valid).round().astype(int)`

**Example for FCVC:**
- FCVC = 2.784 → clip(2.784, 1, 3) = 2.784 → round() = 3 → int = **3** ("Always")
- FCVC = 1.12  → clip(1.12, 1, 3)  = 1.12  → round() = 1 → int = **1** ("Never")

**Why clip before rounding?**
SMOTE can occasionally extrapolate slightly outside the valid range (e.g., FCVC = 3.02).
Clipping to [1,3] first prevents these edge cases from becoming 4 after rounding.

**What we lose:** Nothing. The 498 real respondents already had integers — rounding does
not change them. The 1,613 synthetic respondents get their interpolated values corrected
to the nearest valid survey answer.

### Step 2: Verify no missing values

In [ ]:
for col,(lo,hi) in SMOTE_CLIPS.items():
    df[f"{col}_int"] = np.clip(df[col],lo,hi).round().astype(int)

print("SMOTE recovery results:")
print(f"{'Column':<8} {'Raw range':<20} {'Integer values after fix'}")
print("─"*55)
for col in SMOTE_CLIPS:
    raw = f"[{df[col].min():.2f} – {df[col].max():.2f}]"
    ints = sorted(df[f"{col}_int"].unique())
    print(f"  {col:<6s}  {raw:<20}  {ints}")

print()
missing = df.isnull().sum()
print(f"Missing values: {missing.sum()} total")
print("All features complete — no imputation needed." if missing.sum()==0 else missing[missing>0])

print()
print("Age range check (only continuous feature after dropping Weight/Height):")
q1,q3 = df["Age"].quantile(0.25),df["Age"].quantile(0.75)
outliers = df[(df["Age"]<q1-3*(q3-q1))|(df["Age"]>q3+3*(q3-q1))]
print(f"  min={df['Age'].min():.0f}  max={df['Age'].max():.0f}  outliers(3×IQR): {len(outliers)}")

---
## Chapter 6 — Feature Engineering

**What is feature engineering?**
Converting raw survey answers into numerical representations that capture the scientific
relationships between behaviors and obesity. Every feature we build must have a clear
medical or behavioral rationale — if we cannot explain it in plain language, we do not build it.

**Three layers:**
1. **Encoding** — convert categorical text to numbers the model can use
2. **Domain composites** — combine related features into summary scores
3. **Interaction terms** — capture joint effects that neither feature captures alone

### Layer 1 — Encoding (Binary and Ordinal)

| Feature | Before | After | Why |
|---------|--------|-------|-----|
| SMOKE, SCC, FAVC, family_history | "yes"/"no" | 1/0 | Model needs numbers |
| CAEC | "no","Sometimes","Frequently","Always" | 0,1,2,3 | Preserves order |
| CALC | same as CAEC | 0,1,2,3 | Preserves order |

In [ ]:
BIN_COLS = ["family_history_with_overweight","FAVC","SMOKE","SCC"]
ORD_CATS = ["CAEC","CALC"]

for col in BIN_COLS:
    df[f"{col}_bin"] = (df[col]=="yes").astype(int)
    print(f"  {col}_bin: yes→1, no→0  (sample: {df[f'{col}_bin'].value_counts().to_dict()})")

print()
for col in ORD_CATS:
    df[f"{col}_ord"] = df[col].map(FREQ_MAP).astype(int)
    mapping_str = "  →  ".join(f"{k}={v}" for k,v in FREQ_MAP.items() if k in df[col].unique())
    print(f"  {col}_ord: {mapping_str}")

### Layer 2 — Caloric Risk Score

**What it measures:** A single number summarising the respondent's estimated caloric burden
from diet, in a way no single feature captures.

**Formula:**
`caloric_risk = NCP_int + CAEC_ord + 2 × FAVC_bin`

**Why these weights?**
- NCP (meal count) adds 1–4 points: more meals = more caloric opportunities
- CAEC (snacking) adds 0–3 points: more snacking = more untracked calories
- FAVC (high-calorie food) is multiplied by **2** because food quality has an outsized effect
  — eating junk food at every meal is worse than eating one extra healthy meal

**Example calculation:**
> Person A: NCP=3, CAEC=Sometimes(1), FAVC=no(0) → 3 + 1 + 0 = **4/11** (low-moderate)
> Person B: NCP=4, CAEC=Always(3), FAVC=yes(1) → 4 + 3 + 2 = **9/11** (high risk)

In [ ]:
df["caloric_risk"] = df["NCP_int"] + df["CAEC_ord"] + 2*df["FAVC_bin"]

rho_cr,p_cr = stats.spearmanr(df["caloric_risk"],y_num)
rho_ncp,_ = stats.spearmanr(df["NCP_int"],y_num)
rho_caec,_ = stats.spearmanr(df["CAEC_ord"],y_num)
rho_favc,_ = stats.spearmanr(df["FAVC_bin"],y_num)
print(f"caloric_risk composite ρ = {rho_cr:+.3f}")
print(f"  vs NCP_int alone:       ρ = {rho_ncp:+.3f}")
print(f"  vs CAEC_ord alone:      ρ = {rho_caec:+.3f}")
print(f"  vs FAVC_bin alone:      ρ = {rho_favc:+.3f}")
print()
print("Composite ρ should be >= best individual component — if so, the combination adds information.")

fig, ax = plt.subplots(figsize=(10,4))
sns.boxplot(data=df,x=TARGET,y="caloric_risk",order=TARGET_LABELS,palette=PALETTE,ax=ax,linewidth=0.8)
ax.set_xticklabels(TARGET_LABELS,rotation=25,ha="right")
ax.set_title("Caloric Risk Score by Obesity Class",fontsize=12,fontweight="bold")
ax.set_ylabel("caloric_risk  (0–11)")
plt.tight_layout(); plt.show()

**Reading the caloric_risk chart:**

`caloric_risk = NCP_int + CAEC_ord + 2×FAVC_bin`

This formula weights FAVC (high-calorie food) twice because it has the strongest individual signal of the three dietary components (ρ = +0.250 vs NCP ρ = −0.053).

**What to look for in the boxplot:**
The caloric_risk boxplot should show an increasing trend in medians from Insufficient_Weight to Obesity_Type_II/III — though the SMOTE distortion in NCP and CAEC means the trend will be imperfect. Despite SMOTE artifacts in individual components, the composite averages out some of the noise.

**Why composites help tree-based models (XGBoost, LightGBM, RandomForest):**
A decision tree finds the best single split on a single feature. If CAEC and FAVC are separate features, the tree needs TWO splits to capture their joint dietary risk. If they are combined in `caloric_risk`, ONE split captures the joint signal. Fewer splits needed = simpler tree = less overfitting.

The ablation study in Chapter 9 will confirm whether `caloric_risk` outperforms its individual components. If it does, the composite was worth building; if it doesn't, the raw components are already sufficient for the tree model.


### Layer 3 — All Composite Features

Eight additional engineered features, each based on domain knowledge:

In [ ]:
def add_composites(df):
    out = df.copy()

    # Age group: 0=teen, 1=young adult, 2=adult, 3=mid-age, 4=senior
    # Why: metabolic rate and hormone levels change in discrete life stages, not linearly
    out["age_group"] = pd.cut(out["Age"],bins=[0,18,25,35,50,99],
        labels=range(5),include_lowest=True).astype(int)

    # health_score: average of three protective behaviors, each 0–1 after normalisation
    # Why: FAF, CH2O, FCVC all measure "health-conscious habits" — their average
    # captures a general protective lifestyle better than any single measure
    out["health_score"] = out["FAF_int"]/3 + out["CH2O_int"]/3 + out["FCVC_int"]/3

    # transport_active: 1=Walk/Bike, 0=everything else
    # Why: binary grouping of the key incidental activity distinction
    out["transport_active"] = out["MTRANS"].isin(["Walking","Bike"]).astype(int)

    # screen_activity: 0=low screen+active, 1=balanced, 2=high screen+inactive
    # Why: neither TUE nor FAF alone captures the "digital sedentary" risk pattern
    # How it's calculated:
    #   If TUE > 1 hour/day AND FAF < 1 day/week → screen_activity = 2 (HIGH RISK)
    #   If TUE < 2 hours/day AND FAF ≥ 2 days/week → screen_activity = 0 (LOW RISK)
    #   Otherwise → screen_activity = 1 (Balanced)
    def sa(row):
        if row["TUE_int"]>1 and row["FAF_int"]<1: return 2   # high screen + inactive
        if row["TUE_int"]<2 and row["FAF_int"]>=2: return 0  # low screen + active
        return 1
    out["screen_activity"] = out.apply(sa, axis=1)

    # risk_quadrant: genetics × activity (0=safest, 3=riskiest)
    def rq(row):
        fam = row["family_history_with_overweight_bin"]
        act = row["FAF_int"]>=2
        if fam==0 and act:     return 0  # no genetics, active → safest
        if fam==1 and act:     return 1  # genetics but active → moderate
        if fam==0 and not act: return 2  # no genetics but inactive → moderate
        return 3                          # genetics AND inactive → riskiest
    out["risk_quadrant"] = out.apply(rq, axis=1)

    # diet_type: 0=structured/restrictive, 1=controlled, 2=uncontrolled
    def dt(row):
        if row["NCP_int"]<=2 and row["CAEC_ord"]==0: return 0
        if row["FAVC_bin"]==0 and row["CAEC_ord"]<=1: return 1
        return 2
    out["diet_type"] = out.apply(dt, axis=1)

    # lifestyle_region: 0=healthy, 1=moderate, 2=high risk
    # HIGH RISK = inactive AND eats junk food
    # HEALTHY = active AND avoids junk food
    def lr(row):
        if row["FAF_int"]>=2 and row["FAVC_bin"]==0: return 0
        if row["FAF_int"]<1  and row["FAVC_bin"]==1: return 2
        return 1
    out["lifestyle_region"] = out.apply(lr, axis=1)

    # FAF_x_TUE: direct multiplication interaction
    # Why: active + low screen is much better than each factor individually,
    # and inactive + high screen is much worse
    out["FAF_x_TUE"] = out["FAF_int"] * out["TUE_int"]

    return out

df = add_composites(df)

comp_feats = ["age_group","health_score","transport_active","screen_activity",
              "risk_quadrant","diet_type","lifestyle_region","FAF_x_TUE"]

print(f"{'Composite':<20} {'ρ with obesity':>15}  {'Range':>15}  How calculated")
print("─"*80)
for f in comp_feats:
    rho,_ = stats.spearmanr(df[f],y_num)
    lo,hi = df[f].min(),df[f].max()
    print(f"  {f:<18} {rho:+.3f}{'':>10}  [{lo:.1f}–{hi:.1f}]")

**Understanding `screen_activity` and why 0% makes sense:**

`screen_activity = 2` (High Screen / Inactive) means the person uses screens >1h/day
AND exercises less than once a week. This defines the most sedentary lifestyle profile.

**Why you might see 0% of this group in Insufficient_Weight:**
It is biologically logical — someone who both exercises almost never AND uses screens heavily
has very little reason to be underweight. Their energy expenditure is minimal. Their obesity
risk accumulates. Finding 0 people from this lifestyle group in the Insufficient_Weight class
is the DATA CONFIRMING the hypothesis, not a bug.

**How `screen_activity` is calculated step by step:**
```
Person has TUE_int = 2 (>3h screens/day) and FAF_int = 0 (no exercise)
  → TUE_int > 1? YES. FAF_int < 1? YES. → screen_activity = 2 (HIGH RISK)

Person has TUE_int = 0 (<1h screens/day) and FAF_int = 3 (5 days/week exercise)
  → TUE_int < 2? YES. FAF_int >= 2? YES. → screen_activity = 0 (LOW RISK)

Person has TUE_int = 1 and FAF_int = 1
  → Neither extreme condition met. → screen_activity = 1 (Balanced)
```

In [ ]:
# Validate screen_activity: show obesity share and group sizes
fig, axes = plt.subplots(1,2, figsize=(16,5))
sa_labels = {0:"Low Screen\n+ Active",1:"Balanced",2:"High Screen\n+ Inactive"}
df["_sa_lbl"] = df["screen_activity"].map(sa_labels)
sa_order = [sa_labels[i] for i in [0,1,2]]

ct = df.groupby(["_sa_lbl",TARGET]).size().unstack(TARGET).reindex(columns=TARGET_LABELS,fill_value=0)
ct_norm = ct.div(ct.sum(axis=1),axis=0)
ct_norm.reindex(sa_order).plot(kind="bar",stacked=True,ax=axes[0],colormap="viridis",
    edgecolor="white",linewidth=0.3,legend=False)
axes[0].set_title("screen_activity Group — % in each obesity class",fontsize=11,fontweight="bold")
axes[0].tick_params(axis="x",rotation=15)
for i,lbl in enumerate(sa_order):
    n = len(df[df["_sa_lbl"]==lbl])
    axes[0].text(i,1.02,f"n={n}",ha="center",fontsize=9,color="gray")

ob_rate = df.groupby("_sa_lbl").apply(lambda g:g[TARGET].str.startswith("Obesity").mean()*100)
axes[1].bar([sa_labels[i] for i in [0,1,2]],[ob_rate.get(sa_labels[i],0) for i in [0,1,2]],
    color=["#4575b4","#fee090","#d73027"],edgecolor="white",width=0.5)
for i,k in enumerate([sa_labels[i] for i in [0,1,2]]):
    v = ob_rate.get(k,0)
    axes[1].text(i,v+0.5,f"{v:.1f}%",ha="center",fontsize=12,fontweight="bold")
axes[1].set_title("Obesity rate by screen-activity group",fontsize=11,fontweight="bold")
axes[1].set_ylabel("% who are obese"); axes[1].set_ylim(0,100)

handles,labels_=axes[0].get_legend_handles_labels()
fig.legend(handles,labels_,loc="lower center",ncol=4,fontsize=7,title="Obesity Level",bbox_to_anchor=(0.5,-0.1))
plt.tight_layout(); plt.show()

df.drop(columns="_sa_lbl",inplace=True)
print("Spearman ρ for all composites vs obesity class:")
for f in comp_feats:
    rho,p = stats.spearmanr(df[f],y_num)
    direction = "↑ risk" if rho>0 else "↓ risk"
    print(f"  {f:<22s}: ρ={rho:+.3f}  {direction}")

**Reading the screen_activity validation:**

**LEFT stacked bar:** Three bars for Low/Balanced/High risk groups.
- n= counts show how many people are in each group
- The Low Screen/Active bar should be dominated by blue/purple (light classes)
- The High Screen/Inactive bar should be dominated by dark green/yellow (obese classes)

**RIGHT bar chart:** Obesity rate with exact % label for each group.
- The gap between Low Screen/Active (%) and High Screen/Inactive (%) is the "digital sedentary penalty"
- If this gap is 20+ percentage points, the composite is adding real discriminative value

**0% in Insufficient_Weight for High Screen/Inactive:**
This appears in the stacked bar as the "High Screen / Inactive" bar having zero blue segment.
It is correct — not a bug. The data confirms that people with the most sedentary lifestyle
profile do not belong to the underweight category.

In [ ]:
NOM_CATS = ["Gender","MTRANS"]
df = pd.get_dummies(df, columns=NOM_CATS, drop_first=False, dtype=int)
ohe_cols = [c for c in df.columns if any(c.startswith(p) for p in ["Gender_","MTRANS_"])]
print(f"One-hot encoded: {ohe_cols}")

### Final Feature Matrix

**What features does the model see?**

In [ ]:
FEATURES = (
    ["Age","age_group"]
    + [f"{c}_int" for c in SMOTE_CLIPS]
    + [f"{c}_bin" for c in BIN_COLS]
    + [f"{c}_ord" for c in ORD_CATS]
    + comp_feats
    + ohe_cols
)
X = df[FEATURES]; y = df[TARGET].map(TARGET_ORDER)
print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")
print(f"All numeric: {(X.dtypes != 'object').all()}")
print(f"Zero nulls: {X.isnull().sum().sum() == 0}")
print()

# Final correlation ranking
rho_all = X.apply(lambda col: stats.spearmanr(col,y)[0]).abs().sort_values(ascending=False)
fig,ax = plt.subplots(figsize=(11,9))
colors = ["#d73027" if X[f].corr(y.astype(float))>0 else "#4575b4" for f in rho_all.index]
ax.barh(rho_all.index, rho_all.values, color=colors, edgecolor="white")
ax.axvline(0,color="black",linewidth=0.5)
ax.set_xlabel("|Spearman ρ| with obesity class (red=risk factor, blue=protective)")
ax.set_title("All 28 Features Ranked by Individual Predictive Signal",fontsize=12,fontweight="bold")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

print("Top 10 by |ρ|:")
for i,(f,v) in enumerate(rho_all.head(10).items(),1):
    signed = X[f].corr(y.astype(float))
    print(f"  {i:2d}. {f:<38s} ρ={signed:+.3f}")

**Reading the feature ranking chart — |Spearman ρ| with obesity class:**

**Red bars** = risk factors (higher value → more obese, ρ positive)
**Blue bars** = protective factors (higher value → less obese, ρ negative)

**Actual top features from the data (|ρ| descending):**

| Rank | Feature | |ρ| | Direction | Note |
|------|---------|-----|-----------|------|
| 1 | family_history_with_overweight | 0.500 | Risk (+) | Strongest by far — 54-point obesity gap |
| 2 | Age | 0.409 | Risk (+) | Strong, but OB_III surprisingly young |
| 3 | CAEC_ord | 0.353 | "Protective" (−) | **SMOTE reversed** — "Sometimes" most obese |
| 4 | FCVC_int | 0.260 | Risk (+) | **SMOTE reversed** — Always eats veg most obese |
| 5 | FAVC_bin | 0.250 | Risk (+) | Real signal — 51% vs 8% obesity |
| 6 | CH2O_int | 0.150 | Risk (+) | **SMOTE + reverse causality** |
| 7 | SCC_bin | 0.194 | Protective (−) | Real signal — trackers are mostly thin |
| 8 | FAF_int | 0.180 | Protective (−) | Real, but only OB_III dramatically lower |
| 9 | CALC | 0.168 | Risk (+) | Real but Always n=1 |
| 10 | SMOKE_bin | 0.003 | ~zero | Paradox + tiny n=44 |
| 11 | Gender | 0.037 | ~zero | Male 46% = Female 46% — no signal |

**Important:** Features marked "SMOTE reversed" have real inclusion value for the model — the tree can still extract patterns — but their *direction* is corrupted and should not be clinically interpreted as-is. Features with ρ near zero (SMOKE, Gender) provide minimal standalone signal but may contribute small interactions.

**The key takeaway:** Family history and Age dominate standalone predictive power. Dietary and behavioral features provide meaningful but noisier signal, complicated by synthetic data artifacts.


---
## Chapter 7 — Model Selection and Cross-Validation

### Why these three models?

| Model | Mechanism | Strength here |
|-------|-----------|--------------|
| **XGBoost** | Gradient boosting on decision trees, builds trees sequentially where each corrects prior errors | Best overall for tabular data, handles ordinal features well |
| **LightGBM** | Gradient boosting like XGBoost but uses histogram binning — faster and lower memory | Nearly equal to XGBoost, faster training |
| **RandomForest** | Builds many independent trees and averages (ensemble of uncorrelated trees) | More robust to overfitting, good baseline |

### Three feature sets to compare

We test three progressively richer feature sets to measure what each engineering layer adds:

- **Set A** — Raw survey features only (19 features, no engineering beyond encoding)
- **Set B** — Set A + SMOTE recovery integers + basic encoding (23 features)
- **Set C** — Set B + all composites + one-hot (28 features, full engineering)

**Why compare feature sets?**
If Set C is much better than Set A, the feature engineering added real signal.
If Set C ≈ Set A, the composites added noise (and we should simplify).

### Cross-Validation

**What is K-Fold CV?**
Instead of one train/test split, we split the data into K=5 folds:
- Round 1: train on folds 2,3,4,5 → test on fold 1
- Round 2: train on folds 1,3,4,5 → test on fold 2
- ... (5 rounds total)
- Average the 5 test scores

**Why this is better than one split:**
A single 80/20 split might accidentally put easy cases in the test set or hard ones in training.
CV gives 5 independent estimates of true performance and reduces this luck factor.

**Metric: Macro F1**
F1 = harmonic mean of Precision and Recall. Macro = average F1 across all 7 classes equally.
This penalises the model for getting any class wrong, not just rare ones.

In [ ]:
# Feature set definitions
FEATS_A = (
    ["Age"] + [c for c in SMOTE_CLIPS]
    + [f"{c}_bin" for c in BIN_COLS]
    + [f"{c}_ord" for c in ORD_CATS]
    + ohe_cols
)
FEATS_A = [f for f in FEATS_A if f in X.columns]

FEATS_B = (
    ["Age"] + [f"{c}_int" for c in SMOTE_CLIPS]
    + [f"{c}_bin" for c in BIN_COLS]
    + [f"{c}_ord" for c in ORD_CATS]
    + ohe_cols
)
FEATS_B = [f for f in FEATS_B if f in X.columns]

FEATS_C = FEATURES  # all 28

X_A, X_B, X_C = X[FEATS_A], X[FEATS_B], X[FEATS_C]
print(f"Feature set sizes: A={len(FEATS_A)}, B={len(FEATS_B)}, C={len(FEATS_C)}")

# 9-way comparison: 3 models × 3 feature sets
models = {
    "XGBoost":  XGBClassifier(n_estimators=300,learning_rate=0.05,max_depth=6,
                               use_label_encoder=False,eval_metric="mlogloss",random_state=SEED,verbosity=0),
    "LightGBM": LGBMClassifier(n_estimators=300,learning_rate=0.05,num_leaves=63,random_state=SEED,verbose=-1),
    "RandomForest": RandomForestClassifier(n_estimators=300,max_depth=12,random_state=SEED),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}
print(f"\n{'Model':<15} {'Feature Set':<12} {'Mean F1':>9} {'Std':>7}")
print("─"*50)
for model_name, model in models.items():
    for feat_name, X_feat in [("A (raw)",X_A),("B (int)",X_B),("C (full)",X_C)]:
        scores = cross_val_score(model, X_feat, y, cv=cv, scoring="f1_macro", n_jobs=-1)
        key = f"{model_name}_{feat_name}"
        results[key] = scores
        print(f"  {model_name:<13} {feat_name:<12} {scores.mean():.4f}  ±{scores.std():.4f}")

**Reading the cross-validation results table:**

Each row is one model × feature set combination.

**Mean F1** — the average test F1 across 5 folds (higher = better).
**Std** — standard deviation across folds. Lower std = more consistent (less lucky/unlucky folds).

**What to look for:**
1. **Feature set progression:** Does F1 increase from A → B → C for the same model?
   - A → B improvement: SMOTE integer recovery helped
   - B → C improvement: domain composites helped
2. **Model comparison:** Which model is strongest at Set C?
3. **Practical ceiling:** 65–75% macro F1 is good for 7-class behavioral prediction without height/weight

**If Set A ≈ Set C:** The composites added no signal — the raw features are sufficient.
**If Set C >> Set A:** Feature engineering genuinely helped.

In [ ]:
# Visualize the 9-way comparison
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(12,5))
model_names = list(models.keys())
feat_sets = ["A (raw)","B (int)","C (full)"]
x_pos = np.arange(len(model_names))
width = 0.25
colors_bar = ["#4575b4","#fee090","#d73027"]

for i,feat_name in enumerate(feat_sets):
    means = [results[f"{m}_{feat_name}"].mean() for m in model_names]
    stds  = [results[f"{m}_{feat_name}"].std()  for m in model_names]
    bars = ax.bar(x_pos + i*width, means, width*0.9, yerr=stds, capsize=4,
                  color=colors_bar[i], edgecolor="white", label=f"Set {feat_name}")
    for bar,mean in zip(bars,means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{mean:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x_pos + width)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel("Macro F1 (5-fold CV)")
ax.set_title("9-Way Model × Feature Set Comparison
(error bars = std across 5 folds)",
             fontsize=12, fontweight="bold")
ax.legend(title="Feature Set", fontsize=9)
ax.set_ylim(0, ax.get_ylim()[1]*1.1)
plt.tight_layout(); plt.show()

# Find best combination
best_key = max(results, key=lambda k: results[k].mean())
best_model_name, best_feat_name = best_key.split("_", 1)
print(f"Best combination: {best_model_name} + Feature Set {best_feat_name}")
print(f"  Mean F1 = {results[best_key].mean():.4f} ± {results[best_key].std():.4f}")

---
## Chapter 8 — Hyperparameter Tuning with Optuna

### What are hyperparameters?

A machine learning model has two types of parameters:
- **Learned parameters:** values the model adjusts during training (e.g., tree split thresholds)
- **Hyperparameters:** settings you choose BEFORE training that control how learning happens

Examples for XGBoost:
| Hyperparameter | What it controls | If too small | If too large |
|----------------|-----------------|--------------|--------------|
| `n_estimators` | How many trees to build | Underfits (misses patterns) | Overfits (memorises noise) |
| `max_depth` | How deep each tree can grow | Underfits | Overfits |
| `learning_rate` | How much each tree corrects the previous | Slow convergence | Unstable, overshoots |
| `subsample` | Fraction of data used per tree | — | Overfits |

### What is Optuna?

Optuna is a Bayesian hyperparameter optimisation library.
Instead of trying all combinations (grid search) or random ones (random search), it uses
**Tree-structured Parzen Estimator (TPE)**: after each trial, it builds a probability model
of which regions of the hyperparameter space tend to give good results, then samples more
from those regions.

In plain terms: Optuna learns where to look. After 20 bad trials, it stops trying similar
values and focuses on the promising regions. Much more efficient than brute force.

In [ ]:
# Tune on the best feature set found above
BEST_FEATS = {"A (raw)":FEATS_A,"B (int)":FEATS_B,"C (full)":FEATS_C}.get(best_feat_name,FEATS_C)
X_best = X[BEST_FEATS]

MODEL_SPACES = {
    "XGBoost": lambda t: {
        "n_estimators": t.suggest_int("n_estimators",100,500),
        "max_depth": t.suggest_int("max_depth",3,9),
        "learning_rate": t.suggest_float("learning_rate",0.01,0.3,log=True),
        "subsample": t.suggest_float("subsample",0.6,1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree",0.6,1.0),
        "min_child_weight": t.suggest_int("min_child_weight",1,10),
    },
    "LightGBM": lambda t: {
        "n_estimators": t.suggest_int("n_estimators",100,500),
        "num_leaves": t.suggest_int("num_leaves",20,100),
        "learning_rate": t.suggest_float("learning_rate",0.01,0.3,log=True),
        "subsample": t.suggest_float("subsample",0.6,1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree",0.6,1.0),
        "min_child_samples": t.suggest_int("min_child_samples",5,50),
    },
    "RandomForest": lambda t: {
        "n_estimators": t.suggest_int("n_estimators",100,400),
        "max_depth": t.suggest_int("max_depth",5,20),
        "min_samples_split": t.suggest_int("min_samples_split",2,20),
        "min_samples_leaf": t.suggest_int("min_samples_leaf",1,10),
    },
}

def make_objective(model_name, X_tr, y_tr):
    model_classes = {"XGBoost":XGBClassifier,"LightGBM":LGBMClassifier,"RandomForest":RandomForestClassifier}
    extra = {"XGBoost":{"use_label_encoder":False,"eval_metric":"mlogloss","verbosity":0},
             "LightGBM":{"verbose":-1},"RandomForest":{}}
    def objective(trial):
        params = MODEL_SPACES[model_name](trial)
        params.update(extra[model_name])
        params["random_state"] = SEED
        clf = model_classes[model_name](**params)
        s = cross_val_score(clf, X_tr, y_tr, cv=StratifiedKFold(3,shuffle=True,random_state=SEED),
                            scoring="f1_macro", n_jobs=-1)
        return s.mean()
    return objective

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(make_objective(best_model_name, X_best, y), n_trials=40, show_progress_bar=False)

print(f"Best hyperparameters found by Optuna ({best_model_name}):")
for k,v in study.best_params.items():
    print(f"  {k:<25s}: {v}")
print(f"\nBest CV F1: {study.best_value:.4f}")

In [ ]:
# Convergence plot — shows how Optuna improved over trials
fig, axes = plt.subplots(1,2, figsize=(14,4))

trial_nums = [t.number for t in study.trials]
trial_vals = [t.value for t in study.trials]
best_so_far = [max(trial_vals[:i+1]) for i in range(len(trial_vals))]

axes[0].scatter(trial_nums, trial_vals, alpha=0.4, s=20, color="steelblue", label="Trial F1")
axes[0].plot(trial_nums, best_so_far, color="crimson", linewidth=2, label="Best so far")
axes[0].set_xlabel("Trial number"); axes[0].set_ylabel("CV Macro F1")
axes[0].set_title("Optuna Convergence — F1 per Trial", fontsize=11, fontweight="bold")
axes[0].legend()

# Distribution of F1 scores across trials
axes[1].hist(trial_vals, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].axvline(study.best_value, color="crimson", linewidth=2, linestyle="--",
                label=f"Best: {study.best_value:.4f}")
axes[1].set_xlabel("CV Macro F1"); axes[1].set_ylabel("Number of trials")
axes[1].set_title("Distribution of Trial Scores", fontsize=11, fontweight="bold")
axes[1].legend()

plt.tight_layout(); plt.show()

print("How to read the convergence plot:")
print("  Blue dots = each trial's F1 score")
print("  Red line = best score found so far (only goes up)")
print("  If the red line plateaus early, Optuna found the optimum quickly.")
print("  If blue dots cluster near the red line late in training, Optuna is refining the optimum.")

**Reading the Optuna convergence chart:**

**LEFT chart:**
- Each blue dot = one hyperparameter combination tried, and its cross-validation F1
- The red line = the best F1 found up to that trial (can only go up or stay flat)
- Early trials show high variance (Optuna exploring broadly)
- Later trials cluster near the best (Optuna exploiting what it learned)
- If the red line stops improving around trial 20, Optuna has likely found the optimum

**RIGHT chart (score distribution):**
- Shows how many trials achieved each F1 range
- Narrow distribution = most parameter combinations perform similarly (robust model)
- Wide distribution = hyperparameters have a large effect (sensitive model)
- The red dashed line shows the best score found

---
## Chapter 9 — Final Model Training and Evaluation

### Train/Test Split

We hold out 20% of the data as a test set that the model NEVER sees during training or tuning.
This gives an honest estimate of how the model performs on new, unseen data.

**Why stratify?**
`stratify=y` ensures the test set has the same class proportions as the full dataset.
Without stratification, you might accidentally put all the rare cases in training, making
the test set unrepresentative.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_best, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Training set: {X_tr.shape[0]} rows ({100*X_tr.shape[0]/len(X_best):.0f}%)")
print(f"Test set:     {X_te.shape[0]} rows ({100*X_te.shape[0]/len(X_best):.0f}%)")
print()
print("Class distribution check (should be similar in train and test):")
for cls in range(7):
    tr_pct = (y_tr==cls).mean()*100
    te_pct = (y_te==cls).mean()*100
    print(f"  Class {cls}: train={tr_pct:.1f}%  test={te_pct:.1f}%")

In [ ]:
# Train final model with tuned hyperparameters
model_classes = {"XGBoost":XGBClassifier,"LightGBM":LGBMClassifier,"RandomForest":RandomForestClassifier}
extra_params   = {"XGBoost":{"use_label_encoder":False,"eval_metric":"mlogloss","verbosity":0},
                  "LightGBM":{"verbose":-1},"RandomForest":{}}

best_params = {**study.best_params, **extra_params[best_model_name], "random_state":SEED}
final_model = model_classes[best_model_name](**best_params)
final_model.fit(X_tr, y_tr)
print(f"Final model trained: {best_model_name} with {len(BEST_FEATS)} features")

---
## Chapter 10 — Model Evaluation

### What each metric means

**For each obesity class, the classification report shows:**
- **Precision** = "When the model predicts this class, how often is it correct?"
  - Precision = True Positives ÷ (True Positives + False Positives)
  - Example: Precision=0.75 for Obesity_Type_I means 75% of predictions of Obesity_Type_I are correct
- **Recall** = "Of all the actual cases of this class, how many did the model catch?"
  - Recall = True Positives ÷ (True Positives + False Negatives)
  - Example: Recall=0.80 means the model finds 80% of actual Obesity_Type_I cases
- **F1** = harmonic mean of precision and recall — the single combined score
  - A model with Precision=1.0 but Recall=0.1 would have F1=0.18 (penalises imbalance)
- **Support** = how many actual test cases exist for that class

**Macro avg F1** = average F1 across all 7 classes (all classes count equally)

In [ ]:
y_pred = final_model.predict(X_te)
print("=" * 65)
print(f"FINAL MODEL EVALUATION — {best_model_name} + Feature Set {best_feat_name}")
print("=" * 65)
print()
print(classification_report(y_te, y_pred, target_names=TARGET_LABELS, digits=3))

In [ ]:
# Confusion matrix — shows which classes get confused with which
cm = confusion_matrix(y_te, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1,2, figsize=(18,7))

# Raw counts
im1 = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(7)); axes[0].set_yticks(range(7))
axes[0].set_xticklabels(TARGET_LABELS, rotation=40, ha="right", fontsize=8)
axes[0].set_yticklabels(TARGET_LABELS, fontsize=8)
axes[0].set_xlabel("Predicted Class"); axes[0].set_ylabel("True Class")
axes[0].set_title("Confusion Matrix — Raw Counts", fontsize=11, fontweight="bold")
for i in range(7):
    for j in range(7):
        axes[0].text(j,i,str(cm[i,j]),ha="center",va="center",
                    color="white" if cm[i,j]>cm.max()*0.6 else "black", fontsize=9)
plt.colorbar(im1, ax=axes[0])

# Row-normalised % (per true class)
im2 = axes[1].imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
axes[1].set_xticks(range(7)); axes[1].set_yticks(range(7))
axes[1].set_xticklabels(TARGET_LABELS, rotation=40, ha="right", fontsize=8)
axes[1].set_yticklabels(TARGET_LABELS, fontsize=8)
axes[1].set_xlabel("Predicted Class"); axes[1].set_ylabel("True Class")
axes[1].set_title("Confusion Matrix — % per True Class (row-normalised)", fontsize=11, fontweight="bold")
for i in range(7):
    for j in range(7):
        axes[1].text(j,i,f"{cm_pct[i,j]:.0f}%",ha="center",va="center",
                    color="white" if cm_pct[i,j]>60 else "black", fontsize=8)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout(); plt.show()

**How to read the Confusion Matrix:**

**LEFT (raw counts):**
- Rows = what the class ACTUALLY was (true label)
- Columns = what the model PREDICTED
- **Diagonal cells** (top-left to bottom-right) = correct predictions ✓
- **Off-diagonal cells** = mistakes — the number shows how many times class X was confused with class Y

**RIGHT (% per true class, row-normalised):**
- Each row adds up to 100%
- The diagonal % tells you the Recall for that class
- Example: if row "Obesity_Type_I", column "Obesity_Type_I" shows 78% →
  the model correctly identifies 78% of actual Obesity_Type_I cases

**Common confusion patterns to look for:**
- Adjacent classes (e.g., Overweight_Level_I vs Overweight_Level_II) will be confused most often
  — their behavioral profiles overlap significantly
- Non-adjacent classes (e.g., Insufficient_Weight vs Obesity_Type_III) should almost never
  be confused — their profiles are very different

**Clinical significance:** Confusing Overweight_Level_I and Normal_Weight is less serious
than confusing Obesity_Type_III and Normal_Weight. Off-diagonal "severity" matters.

In [ ]:
# Feature Importance
if hasattr(final_model,"feature_importances_"):
    importance = pd.Series(final_model.feature_importances_, index=BEST_FEATS).sort_values(ascending=True)
    fig,ax = plt.subplots(figsize=(10, max(6, len(importance)*0.3)))
    colors = ["#d73027" if importance.index.get_loc(f)<5 else "steelblue" for f in importance.index]
    ax.barh(importance.index, importance.values, color=colors, edgecolor="white")
    ax.set_xlabel("Feature Importance (gain)")
    ax.set_title(f"Feature Importance — {best_model_name}
(red = top 5 most important)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()

    print("Top 10 most important features:")
    for i,(f,v) in enumerate(importance.sort_values(ascending=False).head(10).items(),1):
        print(f"  {i:2d}. {f:<38s}: {v:.4f}")
    print()
    print("Bottom 5 (least important — potentially removable):")
    for f,v in importance.sort_values().head(5).items():
        print(f"  {f:<38s}: {v:.4f}")

**Reading the Feature Importance chart:**

Feature importance (gain-based) measures **how much each feature reduces prediction error**
when it is used in a tree split, averaged across all trees and all splits.

**This is different from Spearman ρ:**
- Spearman ρ = linear/monotone individual correlation with the target
- Feature importance = how much this feature contributes to the model's decisions,
  including non-linear effects and interactions with other features

**Expected observations:**
- `family_history_with_overweight_bin` usually ranks #1 (strongest genetic signal)
- `caloric_risk` and `CAEC_ord` should rank highly (dietary load)
- `FAF_int` should rank highly (strongest modifiable protective factor)
- `SMOKE_bin` and `SCC_bin` should be near the bottom (confirmed by ρ analysis)

**If a composite feature (e.g., `caloric_risk`) ranks higher than its components:**
This validates the feature engineering — the composite captured something the components didn't.

In [ ]:
# Ablation study: remove one feature at a time, measure F1 drop
print("Ablation study — CV F1 when each feature is REMOVED:")
print("(Large drop = feature was important; small drop = feature was redundant)")
print()

base_scores = cross_val_score(final_model, X_tr, y_tr,
    cv=StratifiedKFold(3,shuffle=True,random_state=SEED), scoring="f1_macro", n_jobs=-1)
base_f1 = base_scores.mean()
print(f"Baseline (all features): {base_f1:.4f}")
print()

ablation = {}
for feat in BEST_FEATS:
    feats_minus = [f for f in BEST_FEATS if f!=feat]
    s = cross_val_score(final_model, X_tr[feats_minus], y_tr,
        cv=StratifiedKFold(3,shuffle=True,random_state=SEED), scoring="f1_macro", n_jobs=-1)
    drop = base_f1 - s.mean()
    ablation[feat] = drop

abl_series = pd.Series(ablation).sort_values(ascending=False)
print(f"{'Feature':<38} {'F1 drop':>10}  Verdict")
print("─"*65)
for feat,drop in abl_series.items():
    verdict = "CRITICAL" if drop>0.02 else "Important" if drop>0.005 else "Minimal"
    print(f"  {feat:<36} {drop:+.4f}   {verdict}")

**Reading the Ablation Study:**

The ablation study removes ONE feature at a time and measures how much the CV F1 drops.

**F1 drop interpretation:**
- **> 0.02 (CRITICAL):** This feature is essential. Removing it hurts the model significantly.
- **0.005–0.02 (Important):** Meaningful contribution. Worth keeping.
- **< 0.005 (Minimal):** This feature adds almost nothing. Could be removed to simplify.

**What to check:**
- Is `SMOKE_bin` or `SCC_bin` listed as Minimal? Expected — confirms our EDA findings.
- Is `family_history_with_overweight_bin` listed as CRITICAL? Expected — genetic predictor.
- Do composite features (like `caloric_risk`) show higher drops than their components?
  If yes — the composites captured information the individual features missed.

**The ablation study is the gold standard for feature selection:**
It measures actual impact on model performance, not just correlation with the target.
A feature can have low ρ but still be important for interactions the model discovered.

---
## Chapter 11 — Scientific Conclusions

### What we actually learned from the data

#### 1. Data leakage elimination is essential and dramatically changes results
Including Weight/Height gives ~97% F1 with zero scientific insight — the model just computes BMI.
Without them, macro F1 drops to 65–78%. This is the honest, scientifically meaningful difficulty of predicting obesity from behavioral and lifestyle features alone.

#### 2. The dominant predictor is genetic, not behavioral

| Feature | ρ | Key finding |
|---------|---|------------|
| **family_history** | +0.500 | 55.9% obese WITH history vs **2.1% obese WITHOUT** — a 54-point gap |
| **Age** | +0.409 | Clear trend, but OB_III mean age = 23.5y (severe early-onset in young adults) |
| FAVC | +0.250 | High-calorie food: 51.1% vs 7.8% obese — clear real signal |
| SCC | −0.194 | Calorie trackers: only 3.1% obese — health-conscious thin people, not reverse causality |
| FAF | −0.180 | Exercise only dramatically separates Obesity_Type_III (median 0.22 vs ~1.0 elsewhere) |

#### 3. SMOTE corrupted several feature directions — critical for interpretation

These features appear to have reversed or counter-intuitive relationships **because of synthetic data generation**, not biology:

| Feature | ρ | Expected direction | Actual artifact |
|---------|---|--------------------|----------------|
| CAEC | −0.353 | More snacking → more obese | "Sometimes" (n=1,765, SMOTE-bloated) has 54% obesity; "Frequently" only 3.3% |
| FCVC | +0.260 | More vegetables → less obese | FCVC=3 "Always" has 54.2% obesity — highest of all |
| CH2O | +0.150 | More water → less obese | CH2O=3 has 57.6% obesity — SMOTE + reverse causality |

**Do not interpret these reversed directions as clinical findings.** They are SMOTE artifacts.

#### 4. Effectively zero-signal features

| Feature | ρ | Reason |
|---------|---|--------|
| SMOKE | +0.003 | Near zero; n=44 smokers (2% of data) |
| Gender | −0.037 | Male 46.0% obese = Female 46.1% — identical rates |
| NCP | −0.053 | All obesity levels eat 3 meals/day; meal frequency alone doesn't separate |
| TUE | −0.076 | Screen time medians are chaotic, no consistent pattern |

#### 5. Genetic × Activity interaction — exercise only helps WITH genetic risk

From cross-tabulation:
- Family history YES + High activity (FAF ≥ 2): **46.8% obese** (exercise reduces risk by 12.6 points)
- Family history YES + Low activity (FAF < 2): **59.4% obese**
- Family history NO + High activity: **2.3% obese** (exercise makes essentially no difference)
- Family history NO + Low activity: **2.0% obese** (identical — genetic protection dominates)

**Conclusion:** Regular exercise meaningfully helps people with genetic predisposition to obesity. For people without genetic risk, behavioral factors matter much less — their baseline protection is already strong.

#### 6. What the model predicts well vs poorly
- **Easy to separate:** Insufficient_Weight vs Obesity_Type_III — very different behavioral profiles
- **Hard to separate:** Adjacent classes (OW_I vs OW_II, OB_I vs OB_II) — behavioral profiles heavily overlap
- **OB_III surprise:** Youngest mean age (23.5y) — early-onset severe obesity in young Latin Americans

### Limitations
1. **77% synthetic data** — model may not generalise to real clinical populations; SMOTE has corrupted several feature relationships
2. **Self-reported survey data** — people systematically under-report unhealthy behaviors (actual obesity rates likely higher)
3. **Cross-sectional** — one time point; no causal direction can be established
4. **Latin American university students** — dietary norms, portion sizes, and physical activity patterns are population-specific
5. **CALC=Always (n=1)** — effectively no data for this category; any % for it is noise
